# Notebook 2 -- Preprocessing enrichi (V2)

## Objectif

Ce notebook contient le **preprocessing enrichi** qui recupere les colonnes a forte valeur predictive ecartees lors du premier pretraitement. Il charge directement `echantillon.csv` (45 colonnes) et exporte `donnees_preprocessees_v2.csv` pret pour la modelisation.

### Differences avec le preprocessing V1 (notebook.ipynb)

| Aspect | V1 (notebook.ipynb) | V2 (ce notebook) |
|---|---|---|
| **Colonnes utilisees** | 14 sur 45 | 26+ sur 45 |
| **OrgId / DetectorId** | Ecartees | Frequency encoding |
| **DeviceId** | Ecarte | Frequency encoding |
| **AlertTitle** | Ecartee | Top 100 + Other |
| **RegistryKey/ValueName** | Ecartees | Conservees (OHE) |
| **ApplicationId/Name** | Ecartees | OHE / Top 50 |
| **OAuthApplicationId** | Ecartee | Conservee (OHE) |
| **ResourceIdName** | Ecartee | Conservee (OHE) |
| **FileName** | Ecartee | Top 50 + Other |
| **FolderPath** | Ecartee | Top 50 + Other |
| **Features d'agregation** | Aucune | nb_alertes, nb_entites, nb_types_entites, nb_categories |

### Justification (article Microsoft)

Freitas et al. (2024) -- *"AI-Driven Guided Response for SOCs with Microsoft Copilot for Security"* (arXiv:2407.09017) -- montrent que **OrgId** et **DetectorId** sont des features critiques dans leur Random Forest de production (Macro F1 = 0.87). Le pipeline Microsoft inclut 5 features categorielles publiques + 67 features numeriques non publiees (proprietaires).

### Cible

Classification 3 classes : `BenignPositive (0)`, `FalsePositive (1)`, `TruePositive (2)`

## 1. Importation des bibliotheques

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

CIBLE_MAP = {'BenignPositive': 0, 'FalsePositive': 1, 'TruePositive': 2}
CIBLE_MAP_INV = {v: k for k, v in CIBLE_MAP.items()}

print("Bibliotheques importees.")

Bibliotheques importees.


## 2. Chargement du dataset brut

On charge directement `echantillon.csv` (dataset brut, 45 colonnes) pour appliquer un preprocessing enrichi qui recupere les colonnes a forte valeur predictive.

In [2]:
raw = pd.read_csv('echantillon.csv', low_memory=False)
print(f"Dataset brut : {raw.shape[0]:,} lignes x {raw.shape[1]} colonnes")
print(f"\nColonnes disponibles ({raw.shape[1]}) :")
print(list(raw.columns))
print(f"\nDistribution de la cible (IncidentGrade) :")
print(raw['IncidentGrade'].value_counts().to_string())
print(f"\nApercu :")
raw.head(3)

Dataset brut : 94,638 lignes x 45 colonnes

Colonnes disponibles (45) :
['Id', 'OrgId', 'IncidentId', 'AlertId', 'Timestamp', 'DetectorId', 'AlertTitle', 'Category', 'MitreTechniques', 'IncidentGrade', 'ActionGrouped', 'ActionGranular', 'EntityType', 'EvidenceRole', 'DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName', 'NetworkMessageId', 'EmailClusterId', 'RegistryKey', 'RegistryValueName', 'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 'ThreatFamily', 'FileName', 'FolderPath', 'ResourceIdName', 'ResourceType', 'Roles', 'OSFamily', 'OSVersion', 'AntispamDirection', 'SuspicionLevel', 'LastVerdict', 'CountryCode', 'State', 'City']

Distribution de la cible (IncidentGrade) :
IncidentGrade
BenignPositive    41105
TruePositive      33220
FalsePositive     20313

Apercu :


,Id,OrgId,IncidentId,AlertId,Timestamp,DetectorId,AlertTitle,Category,MitreTechniques,IncidentGrade,...,ResourceType,Roles,OSFamily,OSVersion,AntispamDirection,SuspicionLevel,LastVerdict,CountryCode,State,City
0,884763264609,21,111280,263403,2024-06-09T11:29:01.000Z,3,4,SuspiciousActivity,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,Suspicious,Suspicious,242,1445,10630
1,515396079824,36,30247,97984,2024-06-05T09:54:14.000Z,10,8,InitialAccess,T1566.001,BenignPositive,...,NaN,NaN,5,66,NaN,NaN,NoThreatsFound,242,1445,10630
2,1511828491965,12,2420,1652,2024-05-23T01:55:44.000Z,16,186,Impact,NaN,BenignPositive,...,NaN,NaN,5,66,NaN,NaN,NaN,242,1445,10630


## 3. Preprocessing enrichi (V2)

### Pipeline de traitement :
1. **Timestamp** -> features temporelles (Hour, DayOfWeek, DayOfMonth, Month, IsWeekend, IsBusinessHour)
2. **OrgId, DetectorId, DeviceId** -> frequency encoding (features critiques selon Microsoft)
3. **AlertTitle** -> top 100 + Other
4. **FileName** -> top 50 + Other
5. **FolderPath** -> top 50 + Other
6. **MitreTechniques** -> technique principale + top 15
7. **NaN** -> remplissage contextuel
8. **Colonnes geographiques** -> reduction cardinalite
9. **Colonnes faible cardinalite** -> conservees pour OHE (RegistryKey, RegistryValueName, etc.)
10. **ApplicationName** -> top 50 + Other
11. **Features d'agregation** -> nb_alertes, nb_entites, nb_types_entites, nb_categories par incident
12. **Suppression** -> identifiants purs + colonnes >97% NaN

In [3]:
df = raw.copy()

# ============================================================
# 3.1 Traitement du Timestamp -> features temporelles
# ============================================================
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df['Hour']       = df['Timestamp'].dt.hour
df['DayOfWeek']  = df['Timestamp'].dt.dayofweek
df['DayOfMonth'] = df['Timestamp'].dt.day
df['Month']      = df['Timestamp'].dt.month
df['IsWeekend']      = (df['DayOfWeek'] >= 5).astype(int)
df['IsBusinessHour'] = df['Hour'].between(8, 18).astype(int)
df.drop(columns=['Timestamp'], inplace=True)
print("[OK] Features temporelles creees : Hour, DayOfWeek, DayOfMonth, Month, IsWeekend, IsBusinessHour")

# ============================================================
# 3.2 Frequency Encoding pour colonnes haute cardinalite
# (OrgId, DetectorId, DeviceId -- features critiques selon Microsoft)
# ============================================================
for col in ['OrgId', 'DetectorId', 'DeviceId']:
    freq = df[col].value_counts(normalize=True)
    df[f'{col}_freq'] = df[col].map(freq).astype(float)
    df.drop(columns=[col], inplace=True)
print("[OK] Frequency encoding : OrgId_freq, DetectorId_freq, DeviceId_freq")

# ============================================================
# 3.3 AlertTitle -> Top 100 + Other
# ============================================================
top_alerts = df['AlertTitle'].value_counts().head(100).index
df['AlertTitle_group'] = df['AlertTitle'].where(df['AlertTitle'].isin(top_alerts), other='Other')
df.drop(columns=['AlertTitle'], inplace=True)
print(f"[OK] AlertTitle -> AlertTitle_group ({df['AlertTitle_group'].nunique()} categories)")

# ============================================================
# 3.4 FileName -> Top 50 + Other
# ============================================================
top_files = df['FileName'].value_counts().head(50).index
df['FileName_group'] = df['FileName'].where(df['FileName'].isin(top_files), other=-1)
df['FileName_group'] = df['FileName_group'].astype(str)
df.drop(columns=['FileName'], inplace=True)
print(f"[OK] FileName -> FileName_group ({df['FileName_group'].nunique()} categories)")

# ============================================================
# 3.5 FolderPath -> Top 50 + Other
# ============================================================
top_folders = df['FolderPath'].value_counts().head(50).index
df['FolderPath_group'] = df['FolderPath'].where(df['FolderPath'].isin(top_folders), other=-1)
df['FolderPath_group'] = df['FolderPath_group'].astype(str)
df.drop(columns=['FolderPath'], inplace=True)
print(f"[OK] FolderPath -> FolderPath_group ({df['FolderPath_group'].nunique()} categories)")

# ============================================================
# 3.6 MitreTechniques -> extraction technique principale + Top 15
# ============================================================
df['MitreTechniques'] = df['MitreTechniques'].fillna('Unknown')
df['MitreTechnique_Main'] = df['MitreTechniques'].apply(
    lambda x: x.split(';')[0].strip() if isinstance(x, str) and x != 'Unknown' else 'Unknown'
)
top_techniques = df['MitreTechnique_Main'].value_counts().head(15).index
df['MitreTechnique_Main'] = df['MitreTechnique_Main'].where(
    df['MitreTechnique_Main'].isin(top_techniques), other='Other'
)
df.drop(columns=['MitreTechniques'], inplace=True)
print(f"[OK] MitreTechniques -> MitreTechnique_Main ({df['MitreTechnique_Main'].nunique()} categories)")

# ============================================================
# 3.7 Traitement des NaN
# ============================================================
df['SuspicionLevel'] = df['SuspicionLevel'].fillna('None')
df['LastVerdict']    = df['LastVerdict'].fillna('Unknown')
df['OSVersion']      = df['OSVersion'].fillna('Unknown')
df['OSFamily']       = df['OSFamily'].fillna('Unknown')
print("[OK] NaN traites : SuspicionLevel->'None', LastVerdict->'Unknown', OSVersion/OSFamily->'Unknown'")

# ============================================================
# 3.8 Reduction de cardinalite pour colonnes geographiques
# ============================================================
def reduire_cardinalite(series, top_n=20, fill='Other'):
    top_vals = series.value_counts().head(top_n).index
    return series.where(series.isin(top_vals), other=fill)

df['City']        = reduire_cardinalite(df['City'], top_n=20)
df['State']       = reduire_cardinalite(df['State'], top_n=20)
df['CountryCode'] = reduire_cardinalite(df['CountryCode'], top_n=30)
df['OSVersion']   = reduire_cardinalite(df['OSVersion'], top_n=15)
print("[OK] Reduction cardinalite : City(20), State(20), CountryCode(30), OSVersion(15)")

# ============================================================
# 3.9 Colonnes a faible cardinalite recuperees (OHE direct)
# RegistryKey (55), RegistryValueName (28), ApplicationId (68),
# OAuthApplicationId (21), ResourceIdName (58)
# -> conservees telles quelles, elles seront OHE dans le pipeline
# ============================================================
top_apps = df['ApplicationName'].value_counts().head(50).index
df['ApplicationName'] = df['ApplicationName'].where(
    df['ApplicationName'].isin(top_apps), other='Other'
)
print(f"[OK] Colonnes faible cardinalite gardees : RegistryKey, RegistryValueName, ApplicationId, OAuthApplicationId, ResourceIdName")
print(f"     ApplicationName -> top 50 ({df['ApplicationName'].nunique()} categories)")

# ============================================================
# 3.10 Features d'agregation par incident
# ============================================================
incident_stats = df.groupby('IncidentId').agg(
    nb_alertes=('AlertId', 'nunique'),
    nb_entites=('Id', 'nunique'),
    nb_types_entites=('EntityType', 'nunique'),
    nb_categories=('Category', 'nunique')
).reset_index()

df = df.merge(incident_stats, on='IncidentId', how='left')
print(f"[OK] Features d'agregation creees : nb_alertes, nb_entites, nb_types_entites, nb_categories")

# ============================================================
# 3.11 Suppression des colonnes identifiants purs et vides
# ============================================================
cols_to_drop = [
    'Id', 'IncidentId', 'AlertId',         # Identifiants purs
    'ActionGrouped', 'ActionGranular',       # 100% NaN
    'EmailClusterId',                        # 99% NaN
    'ThreatFamily',                          # 99.2% NaN
    'ResourceType',                          # 99.9% NaN
    'Roles',                                 # 97.6% NaN
    'AntispamDirection',                     # 98.1% NaN
    'Sha256', 'IpAddress', 'Url',           # Trop granulaires
    'AccountSid', 'AccountUpn',              # Identifiants utilisateurs
    'AccountObjectId', 'AccountName',        # Identifiants utilisateurs
    'DeviceName', 'NetworkMessageId',        # Identifiants nominaux
]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f"[OK] Colonnes supprimees : {len(cols_to_drop)} (identifiants purs + NaN >97%)")

# ============================================================
# 3.12 Conversion des types pour le pipeline
# ============================================================
int_to_cat = ['RegistryKey', 'RegistryValueName', 'RegistryValueData',
              'ApplicationId', 'OAuthApplicationId', 'ResourceIdName',
              'OSFamily', 'OSVersion', 'CountryCode', 'State', 'City']
for col in int_to_cat:
    if col in df.columns:
        df[col] = df[col].astype(str)

df['AlertTitle_group'] = df['AlertTitle_group'].astype(str)
df['ApplicationName'] = df['ApplicationName'].astype(str)

print(f"\n{'='*60}")
print(f"DATASET ENRICHI : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"{'='*60}")
print(f"\nColonnes ({df.shape[1]}) : {list(df.columns)}")

[OK] Features temporelles creees : Hour, DayOfWeek, DayOfMonth, Month, IsWeekend, IsBusinessHour
[OK] Frequency encoding : OrgId_freq, DetectorId_freq, DeviceId_freq
[OK] AlertTitle -> AlertTitle_group (101 categories)
[OK] FileName -> FileName_group (51 categories)
[OK] FolderPath -> FolderPath_group (51 categories)
[OK] MitreTechniques -> MitreTechnique_Main (16 categories)
[OK] NaN traites : SuspicionLevel->'None', LastVerdict->'Unknown', OSVersion/OSFamily->'Unknown'
[OK] Reduction cardinalite : City(20), State(20), CountryCode(30), OSVersion(15)
[OK] Colonnes faible cardinalite gardees : RegistryKey, RegistryValueName, ApplicationId, OAuthApplicationId, ResourceIdName
     ApplicationName -> top 50 (51 categories)
[OK] Features d'agregation creees : nb_alertes, nb_entites, nb_types_entites, nb_categories
[OK] Colonnes supprimees : 19 (identifiants purs + NaN >97%)

DATASET ENRICHI : 94,638 lignes x 35 colonnes

Colonnes (35) : ['Category', 'IncidentGrade', 'EntityType', 'EvidenceR

## 4. Verification des donnees pretraitees

In [4]:
# Verification : dimensions, types, NaN restants
print(f"Dimensions : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nTypes de colonnes :")
print(df.dtypes.value_counts().to_string())

num_cols_v2 = df.select_dtypes(exclude='object').columns.tolist()
cat_cols_v2 = df.select_dtypes(include='object').columns.tolist()
# Exclure la cible
if 'IncidentGrade' in cat_cols_v2:
    cat_cols_v2.remove('IncidentGrade')

print(f"\nNumeriques    ({len(num_cols_v2)}) : {num_cols_v2}")
print(f"Categorielles ({len(cat_cols_v2)}) : {cat_cols_v2}")

nan_total = df.isna().sum().sum()
print(f"\nNaN restants : {nan_total}")
if nan_total > 0:
    nan_cols = df.isna().sum()
    print(nan_cols[nan_cols > 0].sort_values(ascending=False))

print(f"\nDistribution de la cible :")
print(df['IncidentGrade'].value_counts().to_string())

print(f"\nApercu statistique (numeriques) :")
df[num_cols_v2].describe().round(3)

Dimensions : 94,638 lignes x 35 colonnes

Types de colonnes :
object     22
int64       6
int32       4
float64     3

Numeriques    (13) : ['Hour', 'DayOfWeek', 'DayOfMonth', 'Month', 'IsWeekend', 'IsBusinessHour', 'OrgId_freq', 'DetectorId_freq', 'DeviceId_freq', 'nb_alertes', 'nb_entites', 'nb_types_entites', 'nb_categories']
Categorielles (21) : ['Category', 'EntityType', 'EvidenceRole', 'RegistryKey', 'RegistryValueName', 'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 'ResourceIdName', 'OSFamily', 'OSVersion', 'SuspicionLevel', 'LastVerdict', 'CountryCode', 'State', 'City', 'AlertTitle_group', 'FileName_group', 'FolderPath_group', 'MitreTechnique_Main']

NaN restants : 0

Distribution de la cible :
IncidentGrade
BenignPositive    41105
TruePositive      33220
FalsePositive     20313

Apercu statistique (numeriques) :


,Hour,DayOfWeek,DayOfMonth,Month,IsWeekend,IsBusinessHour,OrgId_freq,DetectorId_freq,DeviceId_freq,nb_alertes,nb_entites,nb_types_entites,nb_categories
count,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000,94638.000
mean,12.156,2.490,9.787,5.910,0.171,0.496,0.014,0.041,0.926,20.835,1.103,2.096,1.301
std,6.787,1.865,6.163,0.289,0.376,0.500,0.024,0.047,0.184,37.550,0.347,1.424,0.788
min,0.000,0.000,1.000,1.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,1.000,1.000
25%,6.000,1.000,5.000,6.000,0.000,0.000,0.001,0.003,0.962,1.000,1.000,1.000,1.000
50%,13.000,2.000,9.000,6.000,0.000,0.000,0.005,0.017,0.962,1.000,1.000,1.000,1.000
75%,18.000,4.000,12.000,6.000,0.000,1.000,0.013,0.063,0.962,19.000,1.000,4.000,1.000
max,23.000,6.000,31.000,12.000,1.000,1.000,0.088,0.140,0.962,278.000,5.000,8.000,11.000


## 5. Encodage de la cible et export

La cible est encodee en 3 classes :
- `BenignPositive` -> 0
- `FalsePositive` -> 1
- `TruePositive` -> 2

Le fichier `donnees_preprocessees_v2.csv` est exporte pour etre utilise par les notebooks de modelisation.

In [5]:
# Encodage de la cible
df['cible'] = df['IncidentGrade'].map(CIBLE_MAP)
df.drop(columns=['IncidentGrade'], inplace=True)

# Remplir les NaN restants dans les colonnes categorielles par 'Unknown'
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna('Unknown')

# Remplir les NaN restants dans les colonnes numeriques par 0
for col in df.select_dtypes(exclude='object').columns:
    if col != 'cible':
        df[col] = df[col].fillna(0)

assert df.isna().sum().sum() == 0, "Il reste des NaN !"
print(f"[OK] Cible encodee : {dict(CIBLE_MAP)}")
print(f"[OK] Aucun NaN restant")

# Export
df.to_csv('donnees_preprocessees_v2.csv', index=False)
print(f"\n[OK] Export : donnees_preprocessees_v2.csv")
print(f"     {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"\nDistribution de la cible :")
print(df['cible'].map(CIBLE_MAP_INV).value_counts().to_string())
print(f"\nColonnes finales ({df.shape[1]}) :")
print(list(df.columns))

[OK] Cible encodee : {'BenignPositive': 0, 'FalsePositive': 1, 'TruePositive': 2}
[OK] Aucun NaN restant

[OK] Export : donnees_preprocessees_v2.csv
     94,638 lignes x 35 colonnes

Distribution de la cible :
cible
BenignPositive    41105
TruePositive      33220
FalsePositive     20313

Colonnes finales (35) :
['Category', 'EntityType', 'EvidenceRole', 'RegistryKey', 'RegistryValueName', 'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 'ResourceIdName', 'OSFamily', 'OSVersion', 'SuspicionLevel', 'LastVerdict', 'CountryCode', 'State', 'City', 'Hour', 'DayOfWeek', 'DayOfMonth', 'Month', 'IsWeekend', 'IsBusinessHour', 'OrgId_freq', 'DetectorId_freq', 'DeviceId_freq', 'AlertTitle_group', 'FileName_group', 'FolderPath_group', 'MitreTechnique_Main', 'nb_alertes', 'nb_entites', 'nb_types_entites', 'nb_categories', 'cible']

[OK] Export : donnees_preprocessees_v2.csv
     94,638 lignes x 35 colonnes

Distribution de la cible :
cible
BenignPositive    41105
TruePo